# Evasão e desempenho acadêmico — exploração e modelagem

**Pergunta:** Qual é a relação entre o desempenho acadêmico no primeiro semestre e a situação acadêmica posterior, e quanto esse desempenho acrescenta à capacidade preditiva?

Execute as células em ordem. Este notebook não contém métricas inventadas: os resultados aparecem somente após a execução. O teste fica reservado para avaliação final; a busca de hiperparâmetros ocorre no treino.

In [ ]:
%pip install -q ucimlrepo pandas numpy scikit-learn matplotlib seaborn

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

SEED = 42


## 1. Carregar e inspecionar os dados
A base é obtida do UCI Machine Learning Repository (ID 697).

In [ ]:
dataset = fetch_ucirepo(id=697)
X = dataset.data.features.copy()
y = dataset.data.targets.iloc[:, 0].astype(str).str.strip()
X.columns = [str(c).strip() for c in X.columns]
print("Dimensão dos preditores:", X.shape)
print("Classes:", y.value_counts())
display(X.head())
display(dataset.metadata)

In [ ]:
display(pd.DataFrame({
    "tipo": X.dtypes.astype(str),
    "ausentes": X.isna().sum(),
    "valores_unicos": X.nunique(dropna=False)
}).sort_values("ausentes", ascending=False))
print("Duplicatas exatas (incluindo alvo):", int(X.join(y.rename("TARGET")).duplicated().sum()))

## 2. Exploração visual
As comparações abaixo são descritivas; não demonstram causalidade.

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x=y, order=y.value_counts().index)
plt.title("Distribuição da situação acadêmica")
plt.xlabel("Situação")
plt.ylabel("Quantidade de estudantes")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
first_sem = [c for c in X.columns if c.startswith((
    "Curricular units 1st sem (credited)",
    "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 1st sem (without evaluations)",
))]
print("Indicadores de 1º semestre:", first_sem)
if first_sem:
    descriptive = X[first_sem].copy()
    descriptive["Situação"] = y.values
    display(descriptive.groupby("Situação").mean(numeric_only=True).T)
    grade_cols = [c for c in first_sem if "grade" in c.lower()]
    for col in grade_cols:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=pd.DataFrame({col: X[col], "Situação": y}), x="Situação", y=col)
        plt.title(f"{col} por situação acadêmica")
        plt.tight_layout()
        plt.show()
else:
    print("Revise os nomes das colunas e a documentação da base.")

## 3. Definir os dois cenários

- **A — ingresso:** somente características disponíveis no momento de entrada.
- **B — ingresso + primeiro semestre:** acrescenta os indicadores do primeiro semestre.
- Variáveis do segundo semestre são excluídas para reduzir o risco de vazamento temporal.

A seleção de atributos deve ser conferida com o dicionário oficial da base.

In [ ]:
entry_candidates = [
    "Marital status", "Application mode", "Application order", "Course",
    "Daytime/evening attendance", "Previous qualification",
    "Previous qualification (grade)", "Nacionality", "Nationality",
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation", "Admission grade",
    "Displaced", "Educational special needs", "Debtor",
    "Tuition fees up to date", "Gender", "Scholarship holder",
    "Age at enrollment", "International", "Unemployment rate",
    "Inflation rate", "GDP"
]
entry_cols = [c for c in entry_candidates if c in X.columns]
second_sem = [c for c in X.columns if c.startswith("Curricular units 2nd sem")]
scenario_a = X[entry_cols].copy()
scenario_b = X[list(dict.fromkeys(entry_cols + first_sem))].copy()
print("Atributos cenário A:", entry_cols)
print("Atributos de 1º semestre:", first_sem)
print("Atributos de 2º semestre excluídos:", second_sem)
print("Dimensões A/B:", scenario_a.shape, scenario_b.shape)

## 4. Preparação e modelos
Os transformadores são ajustados dentro dos pipelines em cada partição de validação, evitando que informações do conjunto de validação influenciem a preparação.

In [ ]:
def make_pipeline(X_fit, model):
    numeric = X_fit.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical = [c for c in X_fit.columns if c not in numeric]
    transformers = []
    if numeric:
        transformers.append(("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric))
    if categorical:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical))
    return Pipeline([
        ("prep", ColumnTransformer(transformers=transformers)),
        ("model", model)
    ])

def specs():
    return {
        "Regressão Logística": (
            LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
            {"model__C": [0.1, 1.0, 10.0]}
        ),
        "Árvore de Decisão": (
            DecisionTreeClassifier(class_weight="balanced", random_state=SEED), None
        ),
        "Random Forest": (
            RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1),
            {"model__n_estimators": [200, 400],
             "model__max_depth": [None, 10, 20],
             "model__min_samples_leaf": [1, 3]}
        ),
        "SVM": (
            SVC(class_weight="balanced", probability=True, random_state=SEED), None
        )
    }


## 5. Treino, validação cruzada e teste
A busca de hiperparâmetros é feita apenas no treino. O teste é avaliado uma vez por modelo após a seleção. A métrica principal é macro-F1; também observamos recall por classe.

In [ ]:
def evaluate_scenario(name, features, y):
    X_train, X_test, y_train, y_test = train_test_split(
        features, y, test_size=0.20, stratify=y, random_state=SEED
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    rows, fitted_models = [], {}
    for model_name, (model, grid) in specs().items():
        pipe = make_pipeline(X_train, model)
        if grid:
            search = GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv,
                                  n_jobs=-1, refit=True, return_train_score=False)
            search.fit(X_train, y_train)
            fitted = search.best_estimator_
            cv_mean = search.best_score_
            cv_std = search.cv_results_["std_test_score"][search.best_index_]
            best_params = search.best_params_
        else:
            cv_results = cross_validate(pipe, X_train, y_train, cv=cv,
                                        scoring="f1_macro", n_jobs=-1)
            cv_mean = np.mean(cv_results["test_score"])
            cv_std = np.std(cv_results["test_score"], ddof=1)
            fitted = pipe.fit(X_train, y_train)
            best_params = "Busca não realizada para este modelo"
        pred = fitted.predict(X_test)
        report = classification_report(y_test, pred, output_dict=True, zero_division=0)
        rows.append({
            "cenário": name, "modelo": model_name,
            "CV macro-F1 média": cv_mean, "CV macro-F1 DP": cv_std,
            "Teste macro-F1": f1_score(y_test, pred, average="macro"),
            "Recall Dropout": report.get("Dropout", {}).get("recall", np.nan),
            "melhores parâmetros": str(best_params)
        })
        fitted_models[model_name] = (fitted, y_test, pred)
        print(f"\n{name} — {model_name}")
        print(classification_report(y_test, pred, zero_division=0))
    return pd.DataFrame(rows), fitted_models

results_a, models_a = evaluate_scenario("A — ingresso", scenario_a, y)
results_b, models_b = evaluate_scenario("B — ingresso + 1º semestre", scenario_b, y)
results = pd.concat([results_a, results_b], ignore_index=True)
display(results.sort_values(["cenário", "CV macro-F1 média"], ascending=[True, False]))

## 6. Matrizes de confusão
Escolha os gráficos relevantes e interprete os erros por classe, especialmente falsos negativos para evasão.

In [ ]:
for scenario_name, models in [("A — ingresso", models_a), ("B — ingresso + 1º semestre", models_b)]:
    for model_name, (fitted, y_test, pred) in models.items():
        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, xticks_rotation=20)
        ax.set_title(f"{scenario_name}: {model_name}")
        plt.tight_layout()
        plt.show()

## 7. Interpretação e limitações

Ao concluir, responda com base nos resultados:
1. O cenário B melhorou o macro-F1 em relação ao cenário A?
2. Qual classe foi mais difícil de prever?
3. Qual modelo apresentou melhor compromisso entre macro-F1 e recall de `Dropout`?
4. Os resultados são estáveis entre as partições de validação?
5. Quais limitações impedem generalizar o modelo para outras instituições?

Não interprete associação como causalidade. Não use o modelo para rotular estudantes nem para negar apoio ou oportunidades. Registre no `DIARIO.md` as decisões, falhas e alterações feitas durante a execução.